# V42 Expanded: LQ45 AI Strategy with Comprehensive Visualization

**Objective**: This notebook investigates the underperformance of the V42 Hybrid Strategy relative to the Markowitz Static benchmark.

### Key Diagnostics:
1. **Performance Discrepancy**: Visualizing the cumulative wealth gap.
2. **Risk-Off Frequency**: How often did the strategy exit to Cash?
3. **Turnover Cost**: Estimating the drag from transaction fees.
4. **Signal Conflict**: Analyzing days where AI and Price signals disagreed.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- 1. Load Data (LQ45) ---
file_path = 'data_lq45_2023_2025.xlsx'
if not os.path.exists(file_path):
    # Fallback or error handling
    print("Data file not found. Please ensure 'data_lq45_2023_2025.xlsx' is in the directory.")
else:
    data = pd.read_excel(file_path, index_col=0, parse_dates=True)
    returns = data.pct_change().dropna()
    market_return = returns.mean(axis=1)
    market_price = (1 + market_return).cumprod() * 100

    print(f"Data Loaded: {len(data)} rows")


In [ ]:
# --- 2. Feature Engineering ---
features = pd.DataFrame(index=returns.index)
features['Vol_20'] = market_return.rolling(window=20).std()
ma5 = market_price.rolling(window=5).mean()
ma20 = market_price.rolling(window=20).mean()
ma50 = market_price.rolling(window=50).mean()
features['Dist_MA20'] = market_price / ma20
features['Dist_MA50'] = market_price / ma50

# Trend Signal
trend_bullish = (market_price > ma50).astype(int)

# Target: MA Slope Direction
target = (ma5.shift(-1) > ma5).astype(int)

features = features.dropna()
target = target.reindex(features.index).fillna(0)
trend_bullish = trend_bullish.reindex(features.index).fillna(0)

# Split Data
train_mask = (features.index.year <= 2024)
test_mask = (features.index.year == 2025)

X_train, y_train = features.loc[train_mask], target.loc[train_mask]
X_test = features.loc[test_mask]

# Model Training
xgb_model = xgb.XGBClassifier(n_estimators=120, learning_rate=0.04, max_depth=6, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Predictions
ai_probs = pd.Series(xgb_model.predict_proba(X_test)[:, 1], index=X_test.index)
test_trend = trend_bullish.loc[X_test.index]
opt_t = 0.60


In [ ]:
# --- 3. Run Diagnostics ---

risk_off_mask = (ai_probs < opt_t) & (test_trend == 0)
risk_on_mask = ~risk_off_mask

days_invested = risk_on_mask.sum()
days_cash = risk_off_mask.sum()
pct_cash = days_cash / len(X_test)

ai_low = (ai_probs < opt_t).sum()
trend_low = (test_trend == 0).sum()

print(f"--- DIAGNOSTICS (2025) ---")
print(f"Total Trading Days: {len(X_test)}")
print(f"Days Invested: {days_invested} ({1-pct_cash:.1%})")
print(f"Days in Cash: {days_cash} ({pct_cash:.1%})")
print(f"  - AI Signal Low (< {opt_t}): {ai_low} days")
print(f"  - Price Trend Low (< MA50): {trend_low} days")
print(f"  - CONFLUENCE (Risk Off Trigger): {days_cash} days")


In [ ]:
# --- 4. Turnover & Fee Analysis ---

def optimize_markowitz(selected_returns):
    if len(selected_returns.columns) == 0: return {}
    mu, sigma = selected_returns.mean() * 252, selected_returns.cov() * 252
    sigma += np.eye(len(sigma)) * 1e-4 
    try:
        res = minimize(lambda w: -(np.sum(w*mu)/(np.sqrt(np.dot(w.T, np.dot(sigma, w))) + 1e-6)), 
                       [1./len(mu)]*len(mu), method='SLSQP', 
                       bounds=tuple((0, 1) for _ in range(len(mu))), 
                       constraints=({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}))
        return dict(zip(selected_returns.columns, res.x)) if res.success else {}
    except:
        return {}

# Static Weights for Baseline Comparison
test_dates = X_test.index
loc_idx = returns.index.get_loc(test_dates[0])
window_rets = returns.iloc[loc_idx-30:loc_idx]
static_weights = optimize_markowitz(window_rets.dropna(axis=1, how='any'))

# Simulation Loop
val_hybrid = 100.0
val_static = 100.0
fee = 0.0025
prev_weights_h = {}
prev_weights_s = {}
fee_cost_h = 0.0
fee_cost_s = 0.0
cash_switches = 0
market_switches = 0
prev_state = "Invested"

history_h, history_s = [], []

for i, date in enumerate(test_dates[:-1]):
    # Determine State
    prob = ai_probs.loc[date]
    is_trend_up = test_trend.loc[date] == 1
    is_risk_off = (prob < opt_t) and (not is_trend_up)

    # Hybrid Strategy Weights
    if is_risk_off:
        curr_weights_h = {'CASH': 1.0}
        curr_state = "Cash"
    else:
        curr_weights_h = static_weights # Simplified assumption: same portfolio when invested
        curr_state = "Invested"

    # State Switching Count
    if prev_state == "Invested" and curr_state == "Cash": cash_switches += 1
    elif prev_state == "Cash" and curr_state == "Invested": market_switches += 1

    # Static Strategy Weights (Always Invested)
    curr_weights_s = static_weights

    # Turnover Cost Calculation
    # Hybrid
    all_assets_h = set(list(curr_weights_h.keys()) + list(prev_weights_h.keys()))
    turnover_h = sum(abs(curr_weights_h.get(a, 0) - prev_weights_h.get(a, 0)) for a in all_assets_h)
    cost_h = val_hybrid * turnover_h * fee
    fee_cost_h += cost_h
    val_hybrid -= cost_h

    # Static
    all_assets_s = set(list(curr_weights_s.keys()) + list(prev_weights_s.keys()))
    turnover_s = sum(abs(curr_weights_s.get(a, 0) - prev_weights_s.get(a, 0)) for a in all_assets_s)
    cost_s = val_static * turnover_s * fee
    fee_cost_s += cost_s
    val_static -= cost_s

    # Return Calculation
    next_date = test_dates[i+1]
    # Hybrid
    if 'CASH' in curr_weights_h:
        ret_h = 0
    else:
        ret_h = sum(w * returns.loc[next_date, a] for a, w in curr_weights_h.items())
    
    # Static
    ret_s = sum(w * returns.loc[next_date, a] for a, w in curr_weights_s.items())

    val_hybrid *= (1 + ret_h)
    val_static *= (1 + ret_s)

    history_h.append(val_hybrid)
    history_s.append(val_static)
    
    prev_weights_h = curr_weights_h
    prev_weights_s = curr_weights_s
    prev_state = curr_state

print(f"--- FEE ANALYSIS ---")
print(f"Hybrid Strategy Fee Drag: {fee_cost_h:.2f} points")
print(f"Static Strategy Fee Drag: {fee_cost_s:.2f} points")
print(f"Regime Switches (Invested <-> Cash): {cash_switches + market_switches}")


In [ ]:
# --- 5. Visualizing the Whipsaw Effect ---

plt.figure(figsize=(14, 7))

# 1. Portfolio Value Comparison
plt.subplot(2, 1, 1)
plt.plot(pd.Series(history_h, index=test_dates[:-1]), label='Hybrid Strat (Simulated)', color='green')
plt.plot(pd.Series(history_s, index=test_dates[:-1]), label='Static Strat (Simulated)', color='blue', linestyle='--')
plt.title('Impact of Switching Costs on Performance')
plt.ylabel('Portfolio Value')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Regime Switches
plt.subplot(2, 1, 2)
# Create a step chart for regime
regime_signal = risk_off_mask.astype(int) * -1 # -1 for Cash, 0 for Invested
plt.step(regime_signal.index, regime_signal, where='post', color='red', label='Regime (0=Invested, -1=Cash)')
plt.title('Regime Switches (High Frequency = High Cost)')
plt.yticks([-1, 0], ['Cash', 'Invested'])
plt.xlabel('Date')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v42_diagnostic_whipsaw.png')
plt.show()
